In [66]:
from datasets import load_dataset
dataset = load_dataset("/DATA/disk2/yuhang/.cache/modelscope/datasets/AI-ModelScope/chinese-fineweb-edu-v2",
                       split="train",
                       num_proc=40)

FileNotFoundError: Couldn't find any data file at /DATA/disk2/yuhang/.cache/modelscope/datasets/AI-ModelScope/chinese-fineweb-edu-v2.

In [19]:
print("原始数据集信息:")
print(dataset)
unique_sources = dataset.unique('source')
unique_sources

原始数据集信息:
Dataset({
    features: ['text', 'score', '__index__', 'source'],
    num_rows: 187668844
})


['CCI3',
 'IndustryCorpus2',
 'MiChao',
 'WuDao',
 'TeleChat',
 'SkyPile',
 'wanjuan',
 'ChineseWebText']

#### 将数据集文件根据source类别和score分数进行划分

In [21]:
import os
# Dataset({
#     features: ['text', 'score', '__index__', 'source'],
#     num_rows: 187668844
# })

# 1. 定义保存分类后数据集的根目录
output_dir = '/DATA/disk2/yuhang/.cache/bit_brain_data/chinese-fineweb-classification'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

print(f"\n分类后的数据集将被保存在 '{output_dir}' 目录下。\n")

# 2. 获取所有唯一的 'source' 值
unique_sources = dataset.unique('source')
print(f"找到的唯一 'source' 值: {unique_sources}")

# 3. 外层循环：遍历每个 'source'
for source_value in unique_sources:
    print(f"\n--- 正在处理 Source: '{source_value}' ---")
    
    # 根据 source 创建子目录
    source_dir = os.path.join(output_dir, f"source_{source_value}")
    if not os.path.exists(source_dir):
        os.makedirs(source_dir)
        
    # 过滤出当前 source 的所有数据
    # datasets.filter() 非常高效，它不会立即加载数据到内存
    source_dataset = dataset.filter(lambda example: example['source'] == source_value,
                                    num_proc=40)
    
    # 4. 定义 score 的划分范围 (bins)
    # 根据数据集本身的score分布进行划分
    score_bins = [
        (0.6, 0.75),
        (0.75, 1.0)
    ]
    print(f"  将 'score' 按照预设范围进行划分。")
  
    
    # 5. 内层循环：遍历每个 score 范围
    for i, (lower_bound, upper_bound) in enumerate(score_bins):
        
        # 定义一个清晰的目录名来表示范围
        range_name = f"score_{lower_bound}_to_{upper_bound}"
        
        # 处理最后一个区间的边界，使其包含上界 (e.g., score == 1.0)
        if i == len(score_bins) - 1:
            print(f"  --> 正在处理 Score 范围: [{lower_bound}, {upper_bound}]...")
            # 过滤条件： lower_bound <= score <= upper_bound
            final_filtered_dataset = source_dataset.filter(
                lambda example: lower_bound <= example['score'] <= upper_bound,
                num_proc=40 # 多进程加速
            )
        else:
            print(f"  --> 正在处理 Score 范围: [{lower_bound}, {upper_bound})...")
            # 过滤条件： lower_bound <= score < upper_bound
            final_filtered_dataset = source_dataset.filter(
                lambda example: lower_bound <= example['score'] < upper_bound,
                num_proc=40 # 多进程加速
            )

        # 优化：如果过滤后的数据集为空，则跳过保存
        if len(final_filtered_dataset) == 0:
            print(f"      范围 '{range_name}' 内没有数据，跳过保存。")
            continue
            
        # 6. 将最终过滤出的数据集保存到磁盘
        save_path = os.path.join(source_dir, range_name)
        
        # 检查是否已有数据，若有则可以跳过或覆盖
        if os.path.exists(save_path):
            print(f"      目录 '{save_path}' 已存在，跳过保存。")
            continue
            
        final_filtered_dataset.save_to_disk(save_path)
        print(f"      数据已保存到: '{save_path}'，包含 {len(final_filtered_dataset)} 行。")

print("\n--- 所有分类和保存任务已完成！ ---")

# 如何加载保存的数据？
# a = Dataset.load_from_disk('./classified_data/source_source_A/score_1')
# print("\n加载示例：")
# print(a)


分类后的数据集将被保存在 '/DATA/disk2/yuhang/.cache/bit_brain_data/chinese-fineweb-classification' 目录下。

找到的唯一 'source' 值: ['CCI3', 'IndustryCorpus2', 'MiChao', 'WuDao', 'TeleChat', 'SkyPile', 'wanjuan', 'ChineseWebText']

--- 正在处理 Source: 'CCI3' ---
  将 'score' 按照预设范围进行划分。
  --> 正在处理 Score 范围: [0.6, 0.75)...


Saving the dataset (290/290 shards): 100%|██████████| 28199934/28199934 [07:05<00:00, 66204.60 examples/s] 


      数据已保存到: '/DATA/disk2/yuhang/.cache/bit_brain_data/chinese-fineweb-classification/source_CCI3/score_0.6_to_0.75'，包含 28199934 行。
  --> 正在处理 Score 范围: [0.75, 1.0]...


Saving the dataset (70/70 shards): 100%|██████████| 6793946/6793946 [01:50<00:00, 61334.62 examples/s] 


      数据已保存到: '/DATA/disk2/yuhang/.cache/bit_brain_data/chinese-fineweb-classification/source_CCI3/score_0.75_to_1.0'，包含 6793946 行。

--- 正在处理 Source: 'IndustryCorpus2' ---


...

  将 'score' 按照预设范围进行划分。
  --> 正在处理 Score 范围: [0.6, 0.75)...


Saving the dataset (192/192 shards): 100%|██████████| 18645779/18645779 [04:31<00:00, 68624.65 examples/s] 


      数据已保存到: '/DATA/disk2/yuhang/.cache/bit_brain_data/chinese-fineweb-classification/source_ChineseWebText/score_0.6_to_0.75'，包含 18645779 行。
  --> 正在处理 Score 范围: [0.75, 1.0]...


Saving the dataset (44/44 shards): 100%|██████████| 4199255/4199255 [01:06<00:00, 62847.90 examples/s] 


      数据已保存到: '/DATA/disk2/yuhang/.cache/bit_brain_data/chinese-fineweb-classification/source_ChineseWebText/score_0.75_to_1.0'，包含 4199255 行。

--- 所有分类和保存任务已完成！ ---


#### todo 对分类文件取子集，平衡通用和领域数据
##### 初步判断为领域数据：均选取高分数据
IndustryCorpus2（91G）

#### wanjuan选择题可以适当少放一些
wanjuan (22G)



#### 首先计算划分后各个数据集文件的大小

In [35]:
import os
import math
from pathlib import Path

def get_dir_size(path):
    """
    计算指定路径下所有文件和子文件夹的总大小。
    这是一个递归函数，意味着它会调用自己来处理子文件夹。
    """
    total_size = 0
    # Path(path).rglob('*') 会遍历目录下的所有内容，包括子文件夹里的
    # 这是一种更现代、更简洁的写法
    for file in Path(path).rglob('*'):
        # 确保我们只计算文件的大小
        if file.is_file():
            total_size += file.stat().st_size
    return total_size

def format_size(size_bytes):
    """
    将字节大小格式化为人类易读的形式 (B, KB, MB, GB, TB)。
    """
    if size_bytes == 0:
        return "0B"
    # 定义大小单位的元组
    size_names = ("B", "KB", "MB", "GB", "TB", "PB", "EB", "ZB", "YB")
    # 使用log计算单位的索引
    i = int(math.floor(math.log(size_bytes, 1024)))
    # 计算转换后的大小
    p = math.pow(1024, i)
    s = round(size_bytes / p, 2)
    return f"{s} {size_names[i]}"

# 1. 定义数据集的根目录
# 根据你的日志，数据保存在这个路径下
root_dir = '/DATA/disk2/yuhang/.cache/bit_brain_data/chinese-fineweb-classification'
# 我们只关心高质量数据集
target_score_dir_name = 'score_0.75_to_1.0'

# 用一个字典来存储每个source和它对应的大小
source_sizes = {}

print(f"开始扫描目录: {root_dir}\n")

# 2. 遍历根目录，找到所有的 source_* 文件夹
# os.scandir 比 os.listdir 更高效，因为它在扫描时就获取了文件信息
try:
    for entry in os.scandir(root_dir):
        # 确保它是一个目录并且以 'source_' 开头
        if entry.is_dir() and entry.name.startswith('source_'):
            # 提取 source 的名字，比如从 'source_CCI3' 提取出 'CCI3'
            source_name = entry.name.replace('source_', '')
            
            # 构造我们感兴趣的高质量数据文件夹的完整路径
            high_score_path = os.path.join(entry.path, target_score_dir_name)
            
            # 3. 检查路径是否存在，然后计算大小
            if os.path.exists(high_score_path):
                print(f"正在计算 '{source_name}' 的大小...")
                # 调用我们写的函数来获取文件夹大小
                size = get_dir_size(high_score_path)
                source_sizes[source_name] = size
            else:
                # 如果某个source没有这个分数段的数据，也打印出来，方便我们知晓
                print(f"  - 警告: 路径 '{high_score_path}' 不存在，跳过。")
except FileNotFoundError:
    print(f"错误: 根目录 '{root_dir}' 不存在，请检查路径是否正确。")

# 4. 计算总大小和各自的比例
total_size = sum(source_sizes.values())

print("\n--- 结果分析 ---")
if total_size == 0:
    print("未能计算任何数据的大小，总大小为 0。请检查目录结构和路径是否正确。")
else:
    # 使用我们写的格式化函数，让结果更易读
    print(f"所有 'score_0.75_to_1.0' 数据的总大小: {format_size(total_size)}\n")
    print("各 source 数据大小及其占比:")
    
    # 按照大小从大到小排序，这样结果看起来更有条理
    sorted_sources = sorted(source_sizes.items(), key=lambda item: item[1], reverse=True)
    
    for source, size in sorted_sources:
        # 计算百分比
        proportion = (size / total_size) * 100
        # 使用 f-string 格式化输出，让表格对齐，更美观
        print(f"  - {source:<20}: {format_size(size):<10} ({proportion:.2f}%)")


开始扫描目录: /DATA/disk2/yuhang/.cache/bit_brain_data/chinese-fineweb-classification

正在计算 'wanjuan' 的大小...
正在计算 'MiChao' 的大小...
正在计算 'CCI3' 的大小...
正在计算 'IndustryCorpus2' 的大小...
正在计算 'ChineseWebText' 的大小...
正在计算 'WuDao' 的大小...
正在计算 'SkyPile' 的大小...
正在计算 'TeleChat' 的大小...

--- 结果分析 ---
所有 'score_0.75_to_1.0' 数据的总大小: 280.92 GB

各 source 数据大小及其占比:
  - IndustryCorpus2     : 90.4 GB    (32.18%)
  - CCI3                : 47.64 GB   (16.96%)
  - WuDao               : 42.57 GB   (15.15%)
  - TeleChat            : 29.99 GB   (10.68%)
  - ChineseWebText      : 26.92 GB   (9.58%)
  - wanjuan             : 21.27 GB   (7.57%)
  - SkyPile             : 19.72 GB   (7.02%)
  - MiChao              : 2.41 GB    (0.86%)


In [34]:
from datasets import load_from_disk
wanjuan = load_from_disk("/DATA/disk2/yuhang/.cache/bit_brain_data/chinese-fineweb-classification/source_wanjuan/score_0.6_to_0.75")

print(wanjuan)

print(wanjuan[800])

Dataset({
    features: ['text', 'score', '__index__', 'source'],
    num_rows: 15192682
})
{'text': '该地区自然风光绚丽多姿，为发展旅游经济提供了有利条件，下列风景名胜位于该区域的是（ ）\nA.崂山 蓬莱阁\nB.苏禄王墓 趵突泉\nC.灵岩寺 泰山\nD.孔庙 微山湖\n答案:A解：读图可知，山东半岛位于我国的山东省东部，该地区自然风光绚丽多姿，旅游资源丰富，青岛市的崂山，烟台市的蓬莱阁，都是本区的著名旅游胜地．趵突泉位于济南市境内，泰山位于泰安市境内，孔庙位于济宁市境内．所以位于该区域的风景名胜是崂山、蓬莱阁．根据题意．  \n故选：A．\n解析:\n山东半岛位于我国的山东省东部，气候为温带季风气候，降水在400mm～800mm之间，属于半湿润地区；这里自然风光绚丽多彩，旅游资源丰富，青岛市的崂山，烟台市的蓬莱阁，都是本区的著名旅游胜地．趵突泉位于济南市境内，泰山位于泰安市境内，孔庙位于济宁市境内．  \n【点评】本题考查的是山东省的旅游资源，属于基础题．', 'score': 0.70166015625, '__index__': 171926586, 'source': 'wanjuan'}


In [35]:
# 分析wanjuan数据集中text字段的长度分布
import numpy as np

print("正在分析wanjuan数据集中text字段的长度分布...")

# 计算所有text的长度
text_lengths = []
print("正在计算每条数据的text长度...")

# 遍历数据集计算每条text的长度
for i, example in enumerate(wanjuan):
    text_length = len(example['text'])
    text_lengths.append(text_length)
    
    # 每处理10000条数据显示一次进度
    if (i + 1) % 10000 == 0:
        print(f"  已处理 {i + 1} 条数据...")

# 转换为numpy数组便于统计分析
text_lengths = np.array(text_lengths)

print(f"\n--- wanjuan数据集text长度统计分析 ---")
print(f"数据总条数: {len(text_lengths):,}")
print(f"text长度均值: {text_lengths.mean():.2f} 字符")
print(f"text长度中位数: {np.median(text_lengths):.2f} 字符")
print(f"text长度标准差: {text_lengths.std():.2f} 字符")
print(f"text最短长度: {text_lengths.min()} 字符")
print(f"text最长长度: {text_lengths.max():,} 字符")

# 计算一些百分位数，了解数据分布
percentiles = [25, 50, 75, 90, 95, 99]
print(f"\ntext长度百分位数分布:")
for p in percentiles:
    value = np.percentile(text_lengths, p)
    print(f"  {p}%分位数: {value:.0f} 字符")

# 统计不同长度区间的数据分布
print(f"\ntext长度区间分布:")
bins = [0, 100, 500, 1000, 2000, 5000, 10000, float('inf')]
bin_labels = ['0-100', '100-500', '500-1K', '1K-2K', '2K-5K', '5K-10K', '10K+']

for i in range(len(bins)-1):
    count = np.sum((text_lengths >= bins[i]) & (text_lengths < bins[i+1]))
    percentage = (count / len(text_lengths)) * 100
    print(f"  {bin_labels[i]:<8}: {count:>8,} 条 ({percentage:>5.2f}%)")


正在分析wanjuan数据集中text字段的长度分布...
正在计算每条数据的text长度...
  已处理 10000 条数据...
  已处理 20000 条数据...
  已处理 30000 条数据...
  已处理 40000 条数据...
  已处理 50000 条数据...
  已处理 60000 条数据...
  已处理 70000 条数据...
  已处理 80000 条数据...
  已处理 90000 条数据...
  已处理 100000 条数据...
  已处理 110000 条数据...
  已处理 120000 条数据...
  已处理 130000 条数据...
  已处理 140000 条数据...
  已处理 150000 条数据...
  已处理 160000 条数据...
  已处理 170000 条数据...
  已处理 180000 条数据...
  已处理 190000 条数据...
  已处理 200000 条数据...
  已处理 210000 条数据...
  已处理 220000 条数据...
  已处理 230000 条数据...
  已处理 240000 条数据...
  已处理 250000 条数据...
  已处理 260000 条数据...
  已处理 270000 条数据...
  已处理 280000 条数据...
  已处理 290000 条数据...
  已处理 300000 条数据...
  已处理 310000 条数据...
  已处理 320000 条数据...
  已处理 330000 条数据...
  已处理 340000 条数据...
  已处理 350000 条数据...
  已处理 360000 条数据...
  已处理 370000 条数据...
  已处理 380000 条数据...
  已处理 390000 条数据...
  已处理 400000 条数据...
  已处理 410000 条数据...
  已处理 420000 条数据...
  已处理 430000 条数据...
  已处理 440000 条数据...
  已处理 450000 条数据...
  已处理 460000 条数据...
  已处理 470000 条数据...
  已处理 480000 条数据...


In [2]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B",
                                          use_fast = True)
tokenizer

Qwen2TokenizerFast(name_or_path='Qwen/Qwen3-0.6B', vocab_size=151643, model_max_length=131072, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>', 'additional_special_tokens': ['<|im_start|>', '<|im_end|>', '<|object_ref_start|>', '<|object_ref_end|>', '<|box_start|>', '<|box_end|>', '<|quad_start|>', '<|quad_end|>', '<|vision_start|>', '<|vision_end|>', '<|vision_pad|>', '<|image_pad|>', '<|video_pad|>']}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, normalized

In [70]:
# 读取所有高质量数据集（score_0.75_to_1.0）并合并为一个dataset对象
from datasets import load_from_disk, concatenate_datasets
import os
# 定义数据集的根目录和目标分数目录
root_dir = '/DATA/disk2/yuhang/.cache/bit_brain_data/chinese-fineweb-classification'
target_score_dir_name = 'score_0.75_to_1.0'

# 用来存储所有加载的数据集
datasets_list = []

print(f"开始加载高质量数据集 (score_0.75_to_1.0)...")

# 遍历根目录，找到所有的 source_* 文件夹
try:
    for entry in os.scandir(root_dir):
        # 确保它是一个目录并且以 'source_' 开头
        if entry.is_dir() and entry.name.startswith('source_'):
            # 提取 source 的名字
            source_name = entry.name.replace('source_', '')
            
            # 构造高质量数据文件夹的完整路径
            high_score_path = os.path.join(entry.path, target_score_dir_name)
            
            # 检查路径是否存在
            if os.path.exists(high_score_path):
                print(f"  正在加载 '{source_name}' 数据集...")
                try:
                    # 加载数据集
                    source_dataset = load_from_disk(high_score_path)
                    # 添加source信息到数据集中，方便后续分析
                    #source_dataset = source_dataset.add_column("source", [source_name] * len(source_dataset))
                    datasets_list.append(source_dataset)
                    print(f"    成功加载 {len(source_dataset)} 条数据")
                except Exception as e:
                    print(f"    错误: 无法加载 '{source_name}' 数据集: {e}")
            else:
                print(f"  跳过: '{source_name}' - 路径不存在")

    # 合并所有数据集
    if datasets_list:
        print(f"\n正在合并 {len(datasets_list)} 个数据集...")
        dataset = concatenate_datasets(datasets_list)
        print(f"合并完成！总共包含 {len(dataset)} 条高质量数据")
        
        # 显示数据集的基本信息
        print(f"\n数据集信息:")
        print(f"  - 总样本数: {len(dataset):,}")
        print(f"  - 列名: {dataset.column_names}")
        
    #     # 显示各个source的数据分布
    #     if 'source' in dataset.column_names:
    #         source_counts = {}
    #         for item in dataset:
    #             source = item['source']
    #             source_counts[source] = source_counts.get(source, 0) + 1
            
    #         print(f"\n各数据源分布:")
    #         for source, count in sorted(source_counts.items(), key=lambda x: x[1], reverse=True):
    #             percentage = (count / len(dataset)) * 100
    #             print(f"  - {source:<20}: {count:>10,} ({percentage:.2f}%)")
    # else:
    #     print("错误: 没有找到任何可用的数据集")
    #     dataset = None

except FileNotFoundError:
    print(f"错误: 根目录 '{root_dir}' 不存在，请检查路径是否正确。")
    dataset = None

开始加载高质量数据集 (score_0.75_to_1.0)...
  正在加载 'wanjuan' 数据集...
    成功加载 6414403 条数据
  正在加载 'MiChao' 数据集...
    成功加载 447957 条数据
  正在加载 'CCI3' 数据集...
    成功加载 6793946 条数据
  正在加载 'IndustryCorpus2' 数据集...
    成功加载 13864778 条数据
  正在加载 'ChineseWebText' 数据集...
    成功加载 4199255 条数据
  正在加载 'WuDao' 数据集...
    成功加载 8746771 条数据
  正在加载 'SkyPile' 数据集...
    成功加载 3775098 条数据
  正在加载 'TeleChat' 数据集...
    成功加载 4553685 条数据

正在合并 8 个数据集...
合并完成！总共包含 48795893 条高质量数据

数据集信息:
  - 总样本数: 48,795,893
  - 列名: ['text', 'score', '__index__', 'source']


In [30]:
# 统计数据集中'text'字段的长度信息
if dataset is not None and 'text' in dataset.column_names:
    print("\n正在统计文本长度信息...")
    
    # 计算每个样本的文本长度
    def calculate_text_length(examples):
        """计算文本长度的函数"""
        return {'text_length': [len(text) for text in examples['text']]}
    
    # 使用map函数计算所有样本的文本长度
    dataset_with_length = dataset.map(
        calculate_text_length,
        batched=True,
        num_proc=40,  # 使用多进程加速
        desc="计算文本长度"
    )
    
    # 获取所有文本长度
    text_lengths = dataset_with_length['text_length']
    
    # 计算统计信息
    import numpy as np
    
    mean_length = np.mean(text_lengths)
    median_length = np.median(text_lengths)
    min_length = np.min(text_lengths)
    max_length = np.max(text_lengths)
    std_length = np.std(text_lengths)
    
    print(f"\n文本长度统计信息:")
    print(f"  - 平均长度: {mean_length:.2f} 字符")
    print(f"  - 中位数长度: {median_length:.2f} 字符")
    print(f"  - 最小长度: {min_length} 字符")
    print(f"  - 最大长度: {max_length} 字符")
    print(f"  - 标准差: {std_length:.2f} 字符")
    
    # 计算不同长度区间的分布
    length_ranges = [
        (0, 1000),
        (1000, 2000),
        (2000, 4000),
        (4000, 6000),
    ]
    
    print(f"\n文本长度分布:")
    for min_len, max_len in length_ranges:
        if max_len == float('inf'):
            count = sum(1 for length in text_lengths if length >= min_len)
            range_str = f"{min_len}+ 字符"
        else:
            count = sum(1 for length in text_lengths if min_len <= length < max_len)
            range_str = f"{min_len}-{max_len} 字符"
        
        percentage = (count / len(text_lengths)) * 100
        print(f"  - {range_str:<15}: {count:>10,} ({percentage:.2f}%)")
    
    # 清理临时数据集以节省内存
    del dataset_with_length
    del text_lengths
    
else:
    print("错误: 数据集不存在或缺少'text'字段")



正在统计文本长度信息...


计算文本长度 (num_proc=40): 100%|██████████| 48795893/48795893 [01:26<00:00, 566304.98 examples/s] 



文本长度统计信息:
  - 平均长度: 2900.45 字符
  - 中位数长度: 1518.00 字符
  - 最小长度: 32 字符
  - 最大长度: 4430446 字符
  - 标准差: 6579.91 字符

文本长度分布:
  - 0-1000 字符      : 13,531,455 (27.73%)
  - 1000-2000 字符   : 18,266,407 (37.43%)
  - 2000-4000 字符   : 10,614,860 (21.75%)
  - 4000-6000 字符   :  2,835,941 (5.81%)


In [71]:
dataset

Dataset({
    features: ['text', 'score', '__index__', 'source'],
    num_rows: 48795893
})

In [72]:
# --- 1. 定义超参数 ---
# max_length 保持不变
max_length = 2048
# stride (步长) 是新参数。一个常见的选择是 max_length 的 1/4 或 1/2。
# 这里我们选择 512，意味着块之间有 2048 - 1024 = 1024 个 token 的重叠。
stride = 1024
# 新增：定义要保留的最小块长度，小于此长度的最后一个块将被丢弃
min_chunk_length = 256

# --- 2. 编写新的分词和切块函数 ---
def chunk_and_tokenize_function(examples):
    """
    这个函数接收一批文本 (examples)，然后对每个文本进行分词。
    如果分词后的长度超过 max_length，就使用滑动窗口将其切分成多个块。
    这个函数不仅将长文本切块，还会丢弃掉最后那个过短的块。
    """
    # 首先，对所有文本进行分词，但这次不进行截断和填充，以获取每个文本的完整 token 序列。
    # 我们只关心 'input_ids'
    full_tokenized = tokenizer(examples['text'], truncation=False, padding=False)
    
    # 用来存放最终切分好的所有块
    chunked_input_ids = []
    chunked_attention_mask = []
    
    # 遍历刚刚分词后的每一篇文章
    for input_ids in full_tokenized['input_ids']:
        # 获取当前文章的总长度
        doc_length = len(input_ids)
        
        # 如果文章本身就不够长，就直接填充它，作为一个样本
        if doc_length <= max_length:
            # 填充到 max_length
            padded_ids = input_ids + [tokenizer.pad_token_id] * (max_length - doc_length)
            chunked_input_ids.append(padded_ids)
            # attention_mask 中，真实 token 为 1，填充 token 为 0
            attention_mask = [1] * doc_length + [0] * (max_length - doc_length)
            chunked_attention_mask.append(attention_mask)
        
        # 如果文章太长，就开始滑动窗口切分
        else:
            # 从头开始切
            start_index = 0
            while start_index < doc_length:
                # 定义块的结束位置
                end_index = start_index + max_length
                
                # 从完整序列中切出这个块的 input_ids
                chunk = input_ids[start_index:end_index]
                
                # 如果块的长度小于我们设定的最小阈值，就跳出循环，不再处理这个及之后可能的块
                if len(chunk) < min_chunk_length:
                    break

                # 如果这是最后一个块，且长度不足 max_length，需要填充
                if len(chunk) < max_length:
                    padded_chunk = chunk + [tokenizer.pad_token_id] * (max_length - len(chunk))
                    attention_mask = [1] * len(chunk) + [0] * (max_length - len(chunk))
                # 如果是中间的完整块
                else:
                    padded_chunk = chunk
                    attention_mask = [1] * max_length
                
                chunked_input_ids.append(padded_chunk)
                chunked_attention_mask.append(attention_mask)
                
                # 如果窗口已经滑到了文章末尾，结束循环
                if end_index >= doc_length:
                    break
                    
                # 窗口向前滑动一个步长 (stride)
                start_index += stride

    # 返回一个字典，包含所有新生成的块
    return {
        'input_ids': chunked_input_ids,
        'attention_mask': chunked_attention_mask
    }


# --- 3. 应用新的函数 ---
# 注意：因为一行输入可能产生多行输出，我们必须移除所有原始列，
# 否则会因为行数不匹配而报错。
tokenized_dataset = dataset.map(
    chunk_and_tokenize_function,
    batched=True,           # 依然使用批处理以提高效率
    num_proc=40,            # 依然使用多进程
    remove_columns=dataset.column_names  # 移除所有旧列，非常重要！
)


Map (num_proc=40):  31%|███       | 15041000/48795893 [17:23<40:23, 13930.55 examples/s] Token indices sequence length is longer than the specified maximum sequence length for this model (643714 > 131072). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (183744 > 131072). Running this sequence through the model will result in indexing errors
Map (num_proc=40): 100%|██████████| 48795893/48795893 [1:18:39<00:00, 10340.22 examples/s]


In [ ]:
def tokenize_function(examples):
    return tokenizer(examples['text'],
                     max_length=2048,
                     truncation=True,
                     padding="max_length")

tokenizer_dataset = dataset.map(tokenize_function,
                                num_proc=40,
                                batched=True, 
                                remove_columns=['text']) #! 移除原始文本节省内存

Map (num_proc=40): 100%|██████████| 48795893/48795893 [1:11:02<00:00, 11446.95 examples/s]


In [65]:
tokenized_dataset

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 92550904
})

In [79]:
# 打印前100个样本的input_ids长度
# 计算每个样本中非padding token的长度
def get_non_padding_length(input_ids):
    """计算input_ids中非padding token的数量"""
    # padding token的ID是tokenizer.pad_token_id
    pad_token_id = tokenizer.pad_token_id
    # 计算非padding token的数量
    non_padding_count = sum(1 for token_id in input_ids if token_id != pad_token_id)
    return non_padding_count

print("前100个样本的input_ids长度:")
for i in range(min(10000, len(tokenized_dataset))):
    input_ids_length = get_non_padding_length(tokenized_dataset[i]['input_ids'])
    print(f"样本 {i}: {input_ids_length}")


前100个样本的input_ids长度:
样本 0: 301
样本 1: 271
样本 2: 230
样本 3: 366
样本 4: 369
样本 5: 248
样本 6: 800
样本 7: 137
样本 8: 159
样本 9: 285
样本 10: 430
样本 11: 96
样本 12: 162
样本 13: 159
样本 14: 293
样本 15: 202
样本 16: 346
样本 17: 203
样本 18: 296
样本 19: 133
样本 20: 129
样本 21: 682
样本 22: 226
样本 23: 131
样本 24: 335
样本 25: 171
样本 26: 132
样本 27: 249
样本 28: 288
样本 29: 170
样本 30: 209
样本 31: 188
样本 32: 1014
样本 33: 202
样本 34: 201
样本 35: 280
样本 36: 1542
样本 37: 151
样本 38: 337
样本 39: 168
样本 40: 133
样本 41: 132
样本 42: 295
样本 43: 114
样本 44: 310
样本 45: 118
样本 46: 191
样本 47: 281
样本 48: 275
样本 49: 145
样本 50: 336
样本 51: 141
样本 52: 228
样本 53: 343
样本 54: 157
样本 55: 111
样本 56: 490
样本 57: 596
样本 58: 594
样本 59: 217
样本 60: 204
样本 61: 146
样本 62: 207
样本 63: 371
样本 64: 241
样本 65: 259
样本 66: 542
样本 67: 169
样本 68: 391
样本 69: 789
样本 70: 236
样本 71: 268
样本 72: 108
样本 73: 371
样本 74: 301
样本 75: 253
样本 76: 303
样本 77: 171
样本 78: 162
样本 79: 164
样本 80: 231
样本 81: 159
样本 82: 118
样本 83: 325
样本 84: 118
样本 85: 138
样本 86: 438
样本 87: 298
样本 88: 737
样本 89: 28

In [ ]:
# def tokenize_function(examples):
#     return tokenizer(examples['text'],
#                      max_length=2048,
#                      truncation=True,
#                      padding="max_length")

# tokenizer_dataset = dataset.map(tokenize_function,
#                                 num_proc=40,
#                                 batched=True, 
#                                 remove_columns=['text']) #! 移除原始文本节省内存

Map (num_proc=40): 100%|██████████| 48795893/48795893 [1:11:02<00:00, 11446.95 examples/s]


In [ ]:
# def tokenize_function(examples):
#     return tokenizer(examples['text'],
#                      max_length=2048,
#                      truncation=True,
#                      padding="max_length")

# tokenizer_dataset = dataset.map(tokenize_function,
#                                 num_proc=40,
#                                 batched=True, 
#                                 remove_columns=['text']) #! 移除原始文本节省内存

Map (num_proc=40): 100%|██████████| 48795893/48795893 [1:11:02<00:00, 11446.95 examples/s]


In [73]:
tokenized_dataset

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 73019093
})

In [75]:
tokenizer.decode(tokenized_dataset[0]['input_ids'])

'先秦民本思想的杰出代表、论述“民水君舟”的荀子曾在《王制》篇中毫不含糊地说：“庶人安政，然后君子安位。……君子者，天地之参也，万物之总也，民之父母也。”这反映出荀子（ ）\nA.游离于民本与尊君的两端，成为儒法思想的集大成者\nB.从尊君角度论述民本思想，反映君民关系本质上相通\nC.从民本角度论述尊君思想，适应专制统治强化的需要\nD.区分尊君与民本思想，以说明君主专制反民本的本质\n答案:BB\n解析:\n根据题干中“君子者，天地之参也，万物之总也，民之父母也”可知是荀子是从尊君的角度来论证君民关系的，故B项正确。  \n荀子是儒家思想的代表人物，故A项错误。  \nCD项不符合题意，应排除。  \n故选：B。  \n本题考查百家争鸣。需要掌握荀子的思想主张。解题的关键是对“君子者，天地之参也，万物之总也，民之父母也”的分析理解。  \n本题考查百家争鸣。考查对荀子的思想主张的把握，考查学生抓住关键信息、分析理解、运用所学知识解决问题的能力。<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext

In [ ]:
# def tokenize_function(examples):
#     return tokenizer(examples['text'],
#                      max_length=2048,
#                      truncation=True,
#                      padding="max_length")

# tokenizer_dataset = dataset.map(tokenize_function,
#                                 num_proc=40,
#                                 batched=True, 
#                                 remove_columns=['text']) #! 移除原始文本节省内存

Map (num_proc=40): 100%|██████████| 48795893/48795893 [1:11:02<00:00, 11446.95 examples/s]


In [76]:
print(tokenizer_dataset[4879589].keys())

dict_keys(['input_ids', 'attention_mask'])


In [53]:
tokenizer_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

In [77]:
tokenizer_dataset.format

{'type': 'torch',
 'format_kwargs': {},
 'columns': ['input_ids', 'attention_mask'],
 'output_all_columns': False}

In [78]:
tokenizer_dataset.save_to_disk("/DATA/disk2/yuhang/.cache/bit_brain_data/step3_tokenizer_data",
                               max_shard_size = "2024MB")

Saving the dataset (248/248 shards): 100%|██████████| 48795893/48795893 [08:51<00:00, 91737.43 examples/s] 


In [ ]:
# def tokenize_function(examples):
#     return tokenizer(examples['text'],
#                      max_length=2048,
#                      truncation=True,
#                      padding="max_length")

# tokenizer_dataset = dataset.map(tokenize_function,
#                                 num_proc=40,
#                                 batched=True, 
#                                 remove_columns=['text']) #! 移除原始文本节省内存

Map (num_proc=40): 100%|██████████| 48795893/48795893 [1:11:02<00:00, 11446.95 examples/s]
